In [1]:
import pandas as pd
import numpy as np
path = r"C:\Users\ariel\Desktop\Thesis\F-TM-CR\data\lc_after_02_data_prep\lc_basic_database_+_desc.csv"

# טיפול בעמודת זמן: סוג + עמודות עזר

In [2]:
import pandas as pd

path = r"C:\Users\ariel\Desktop\Thesis\F-TM-CR\data\lc_after_02_data_prep\lc_basic_database_+_desc.csv"

# 1) קריאה מחדש, עם כפייה של issue_d כמחרוזת (כדי לא לאבד את המקור)
df_raw = pd.read_csv(path, dtype={"issue_d": "string"})

# 2) הצצה לערכים המקוריים (עם repr כדי לראות רווחים/תווים נסתרים)
print(df_raw["issue_d"].head(10).map(repr).to_list())

# 3) ניקוי בסיסי + המרה קשיחה
s = (df_raw["issue_d"]
     .str.strip()
     .str.replace("\u00a0", "", regex=False))  # non-breaking space אם קיים

df_work = df_raw.copy(deep=True)
df_work["issue_d"] = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")

# 4) בדיקה מה לא הומר
print("NaT count:", df_work["issue_d"].isna().sum())
print("Bad examples:", s[df_work["issue_d"].isna()].dropna().unique()[:20])


["'2011-12-01'", "'2011-12-01'", "'2011-12-01'", "'2011-12-01'", "'2011-12-01'", "'2011-12-01'", "'2011-12-01'", "'2011-12-01'", "'2011-12-01'", "'2011-12-01'"]
NaT count: 217287
Bad examples: <StringArray>
['2011-12-01', '2011-11-01', '2011-10-01', '2011-09-01', '2011-08-01',
 '2011-07-01', '2011-06-01', '2011-05-01', '2011-04-01', '2011-03-01',
 '2011-02-01', '2011-01-01', '2010-12-01', '2010-11-01', '2010-10-01',
 '2010-09-01', '2010-08-01', '2010-07-01', '2010-06-01', '2010-05-01']
Length: 20, dtype: string


In [3]:
df_work = df_raw.copy(deep=True)

s = df_work["issue_d"].str.strip()
df_work["issue_d"] = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")

print(df_work["issue_d"].dtype)
print("NaT count:", df_work["issue_d"].isna().sum())
print(df_work["issue_d"].head(10))

datetime64[ns]
NaT count: 0
0   2011-12-01
1   2011-12-01
2   2011-12-01
3   2011-12-01
4   2011-12-01
5   2011-12-01
6   2011-12-01
7   2011-12-01
8   2011-12-01
9   2011-12-01
Name: issue_d, dtype: datetime64[ns]


In [4]:
df = df_work

In [5]:
df["issue_month_start"] = df["issue_d"].dt.to_period("M").dt.to_timestamp()
df["issue_ym"] = df["issue_d"].dt.to_period("M").astype(str)

min_m = df["issue_month_start"].min()
df["month_idx"] = (
    (df["issue_month_start"].dt.year - min_m.year) * 12
    + (df["issue_month_start"].dt.month - min_m.month)
)


In [6]:
print(df[["issue_d","issue_month_start","issue_ym","month_idx"]].head(10))
print(df[["issue_month_start","issue_ym","month_idx"]].isna().sum())
print(df["month_idx"].min(), df["month_idx"].max(), df["month_idx"].nunique())


     issue_d issue_month_start issue_ym  month_idx
0 2011-12-01        2011-12-01  2011-12         23
1 2011-12-01        2011-12-01  2011-12         23
2 2011-12-01        2011-12-01  2011-12         23
3 2011-12-01        2011-12-01  2011-12         23
4 2011-12-01        2011-12-01  2011-12         23
5 2011-12-01        2011-12-01  2011-12         23
6 2011-12-01        2011-12-01  2011-12         23
7 2011-12-01        2011-12-01  2011-12         23
8 2011-12-01        2011-12-01  2011-12         23
9 2011-12-01        2011-12-01  2011-12         23
issue_month_start    0
issue_ym             0
month_idx            0
dtype: int64
0 47 48


Business rationale:
loan_amnt represents the requested/contracted loan size, while funded_amnt is the amount actually funded.
In this dataset, funded_amnt is almost always equal to loan_amnt, so keeping both is redundant and may add multicollinearity.
We therefore engineer funded_ratio = funded_amnt / loan_amnt to capture the informative cases of partial funding,
and then drop funded_amnt while keeping loan_amnt + funded_ratio.

In [7]:
# ===== Loan amounts: keep loan_amnt + funded_ratio, drop funded_amnt (near-duplicate) =====
required = {"loan_amnt", "funded_amnt"}
if required.issubset(df.columns):
    # ensure numeric before ratio
    df["loan_amnt"] = pd.to_numeric(df["loan_amnt"], errors="coerce")
    df["funded_amnt"] = pd.to_numeric(df["funded_amnt"], errors="coerce")

    # ratio captures partial funding cases; avoid divide-by-zero
    df["funded_ratio"] = np.where(
        df["loan_amnt"].notna() & (df["loan_amnt"] != 0),
        df["funded_amnt"] / df["loan_amnt"],
        np.nan
    )

    df.drop(columns=["funded_amnt"], inplace=True)  # drop near-duplicate column
    print("OK: Created funded_ratio and dropped funded_amnt (kept loan_amnt).")
else:
    print(f"Warning: Missing columns for funded_ratio: {sorted(required - set(df.columns))}")


OK: Created funded_ratio and dropped funded_amnt (kept loan_amnt).


# לוג וטיפול בחריגים

In [8]:
import numpy as np
import pandas as pd

# =========================
# LOG pipeline: select -> numeric -> check negatives -> cap -> log1p (keep originals)
# =========================

log_candidates = [
    "annual_inc",
    "revol_bal",
    "tot_hi_cred_lim",
    "total_rev_hi_lim",
    "total_bc_limit",
    "bc_open_to_buy",
    "avg_cur_bal",
    "loan_amnt",
]

# keep only existing columns
log_cols = [c for c in log_candidates if c in df.columns]

# coerce to numeric
for c in log_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

CAP_Q = 0.995  # upper cap quantile (adjust if needed)
report = []

for c in log_cols:
    s = df[c]
    nonnull = int(s.notna().sum())
    neg_count = int((s < 0).sum(skipna=True))
    zero_count = int((s == 0).sum(skipna=True))

    if nonnull == 0:
        report.append((c, "SKIPPED (all NaN)", nonnull, zero_count, neg_count, None))
        continue

    if neg_count > 0:
        report.append((c, "SKIPPED (has negatives)", nonnull, zero_count, neg_count, None))
        continue

    cap_val = float(s.quantile(CAP_Q))
    s_capped = s.clip(upper=cap_val)

    df[f"{c}_log1p"] = np.log1p(s_capped)  # create new feature, keep original

    report.append((c, "OK", nonnull, zero_count, neg_count, cap_val))

# summary report
rep = pd.DataFrame(report, columns=["col", "status", "nonnull", "zeros", "negatives", f"cap_q{CAP_Q}"])
print(rep.sort_values(["status","col"]).to_string(index=False))


             col status  nonnull  zeros  negatives  cap_q0.995
      annual_inc     OK   217287      0          0  283426.710
     avg_cur_bal     OK   158613     24          0   87093.920
  bc_open_to_buy     OK   176805   3577          0   79011.720
       loan_amnt     OK   217287      0          0   35000.000
       revol_bal     OK   217287   1305          0  105267.100
  total_bc_limit     OK   178286   1484          0  107443.875
total_rev_hi_lim     OK   158619     70          0  166499.550


In [9]:
# Replace raw amount features with their log1p versions for modeling
log_base_cols = [
    "annual_inc", "avg_cur_bal", "bc_open_to_buy", "loan_amnt",
    "revol_bal", "tot_hi_cred_lim", "total_bc_limit", "total_rev_hi_lim"
]

# keep only pairs that exist
replace_cols = [c for c in log_base_cols if c in df.columns and f"{c}_log1p" in df.columns]

# 1) overwrite the original columns with the log values
for c in replace_cols:
    df[c] = df[f"{c}_log1p"]

# 2) drop the helper log columns (optional; keeps df clean)
df.drop(columns=[f"{c}_log1p" for c in replace_cols], inplace=True)

print("Replaced with log1p and dropped helper columns:", replace_cols)


Replaced with log1p and dropped helper columns: ['annual_inc', 'avg_cur_bal', 'bc_open_to_buy', 'loan_amnt', 'revol_bal', 'total_bc_limit', 'total_rev_hi_lim']


In [10]:
check_cols = ['annual_inc','avg_cur_bal','bc_open_to_buy','loan_amnt','revol_bal',
              'tot_hi_cred_lim','total_bc_limit','total_rev_hi_lim']
check_cols = [c for c in check_cols if c in df.columns]
print(df[check_cols].describe().T[["min","50%","max"]])

                       min        50%        max
annual_inc        8.314097  11.018646  12.554712
avg_cur_bal       0.000000   8.959183  11.374754
bc_open_to_buy    0.000000   8.162516  11.277364
loan_amnt         6.908755   9.392745  10.463132
revol_bal         0.000000   9.394244  11.564266
total_bc_limit    0.000000   9.595671  11.584733
total_rev_hi_lim  0.000000  10.043293  12.022754


# conversions + one hot

In [11]:
# =========================
# 4.1 Convert to numeric (only where relevant)
# =========================

# Add here columns that should be numeric in your dataset
numeric_candidates = [
    # counts / months-since / amounts not already handled
    "int_rate", "dti", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "pub_rec", "total_acc",
    "acc_now_delinq", "tot_coll_amt", "tot_cur_bal",
    "total_pymnt", "total_rec_prncp", "total_rec_int",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_amnt",
    "mths_since_last_delinq", "mths_since_last_record",
    "mths_since_last_major_derog", "mths_since_rcnt_il",
    "mths_since_recent_bc", "mths_since_recent_inq",
    "mths_since_recent_revol_delinq",
    # add more as needed
]

numeric_cols = [c for c in numeric_candidates if c in df.columns]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")  # coercion is safe; bad parses -> NaN

# =========================
# 4.2 Convert to categorical / ordinal
# =========================

# term: " 36 months" / "60 months" -> 36 / 60
if "term" in df.columns:
    df["term"] = df["term"].astype(str).str.extract(r"(\d+)")[0]
    df["term"] = pd.to_numeric(df["term"], errors="coerce").astype("Int64")

# grade / sub_grade: keep as ordered categorical (no one-hot here)
if "grade" in df.columns:
    grade_order = list("ABCDEFG")
    df["grade"] = df["grade"].astype(str).str.strip()
    df.loc[df["grade"].isin(["", "nan", "None"]), "grade"] = np.nan
    df["grade"] = pd.Categorical(df["grade"], categories=grade_order, ordered=True)

if "sub_grade" in df.columns:
    sub_order = [f"{g}{i}" for g in "ABCDEFG" for i in range(1, 6)]
    df["sub_grade"] = df["sub_grade"].astype(str).str.strip()
    df.loc[df["sub_grade"].isin(["", "nan", "None"]), "sub_grade"] = np.nan
    df["sub_grade"] = pd.Categorical(df["sub_grade"], categories=sub_order, ordered=True)


# emp_length: convert to numeric years (0-10), keep Int64 (nullable)
if "emp_length" in df.columns:
    s = df["emp_length"].astype(str).str.lower().str.strip()
    s = (s.replace({"< 1 year": "0", "10+ years": "10", "n/a": np.nan})
           .str.extract(r"(\d+)")[0])
    df["emp_length"] = pd.to_numeric(s, errors="coerce").astype("Int64")

# standard categoricals (keep as 'category' dtype for memory + later encoding)
cat_cols = [c for c in ["addr_state", "purpose", "home_ownership", "verification_status", "zip_code"] if c in df.columns]
for c in cat_cols:
    df[c] = df[c].astype("category")

# Optional: if zip_code is messy, standardize to a clean 3-digit ZIP3-like token
# (keeps privacy; good for grouping/fairness checks)
if "zip_code" in df.columns:
    z = df["zip_code"].astype(str).str.extract(r"(\d{3})")[0]
    df["zip3"] = z.astype("category")  # new clean category


In [12]:
print(df[["term","grade","sub_grade","emp_length"]].dtypes if set(["term","grade","sub_grade","emp_length"]).issubset(df.columns) else "Some cols missing")
print(df[["term","grade","sub_grade","emp_length"]].head(5) if set(["term","grade","sub_grade","emp_length"]).issubset(df.columns) else "")

term             Int64
grade         category
sub_grade     category
emp_length       Int64
dtype: object
   term grade sub_grade  emp_length
0    36     B        B2          10
1    60     C        C4           0
2    36     C        C1          10
3    60     B        B5           1
4    36     A        A4           3


In [13]:
# =========================
# 4.3 earliest_cr_line + initial_list_status (were reaching the CSV raw)
# =========================

# earliest_cr_line: "Jan-1985" -> credit history length (months) at origination.
# The raw date is deliberately dropped: as a number it would just track the
# portfolio's time trend; the difference from issue_d is the risk signal.
if "earliest_cr_line" in df.columns:
    ecl = pd.to_datetime(df["earliest_cr_line"].astype(str).str.strip(),
                         format="%b-%Y", errors="coerce")
    df["credit_history_months"] = (
        (df["issue_d"].dt.year - ecl.dt.year) * 12
        + (df["issue_d"].dt.month - ecl.dt.month)
    ).astype("Int64")

    n_bad = int(df["credit_history_months"].isna().sum())
    n_neg = int((df["credit_history_months"] < 0).sum())
    print(f"credit_history_months: created (parse failures: {n_bad}, negative: {n_neg})")
    if n_neg:  # credit line "opened" after the loan was issued -- data error, not information
        df.loc[df["credit_history_months"] < 0, "credit_history_months"] = pd.NA
        print(f"  {n_neg} negative values set to NA")
    print(df["credit_history_months"].describe())

    df.drop(columns=["earliest_cr_line"], inplace=True)
else:
    print("Warning: earliest_cr_line not found")

# initial_list_status: binary f/w -> single 0/1 flag (no need for full one-hot)
if "initial_list_status" in df.columns:
    vals = set(df["initial_list_status"].dropna().astype(str).str.strip().unique())
    if not vals <= {"f", "w"}:
        raise ValueError(f"initial_list_status has unexpected values: {vals - {'f', 'w'}}")
    df["initial_list_status_w"] = (
        df["initial_list_status"].astype(str).str.strip() == "w"
    ).astype("int8")
    print(f"initial_list_status_w: created "
          f"(w: {int(df['initial_list_status_w'].sum()):,}, "
          f"f: {int((df['initial_list_status_w'] == 0).sum()):,})")
    df.drop(columns=["initial_list_status"], inplace=True)
else:
    print("Warning: initial_list_status not found")


credit_history_months: created (parse failures: 0, negative: 0)
count      217287.0
mean     183.987468
std       84.628485
min            36.0
25%           127.0
50%           168.0
75%           226.0
max           785.0
Name: credit_history_months, dtype: Float64


initial_list_status_w: created (w: 39,381, f: 177,906)

In [14]:
cat_like = [c for c in df.columns if str(df[c].dtype) in ["object", "category"]]

card = (
    pd.DataFrame({
        "col": cat_like,
        "dtype": [str(df[c].dtype) for c in cat_like],
        "nunique": [df[c].nunique(dropna=True) for c in cat_like],
        "null_pct": [(df[c].isna().mean()*100) for c in cat_like],
    })
    .sort_values("nunique", ascending=False)
)

print(card.to_string(index=False))


                col    dtype  nunique  null_pct
          emp_title   object   131133  5.960780
               desc   object    98838 54.077326
              title   object    55567  0.006903
           zip_code category      843  0.000000
               zip3 category      843  0.000000
         addr_state category       49  0.000000
           issue_ym   object       48  0.000000
          sub_grade category       35  0.000000
            purpose category       13  0.000000
              grade category        7  0.000000
     home_ownership category        5  0.000000
verification_status category        3  0.000000


In [15]:
ohe_cols = ["addr_state", "purpose", "home_ownership", "verification_status"]
ohe_cols = [c for c in ohe_cols if c in df.columns]

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True, dtype="int8")
print("One-hot created for:", ohe_cols)


One-hot created for: ['addr_state', 'purpose', 'home_ownership', 'verification_status']


In [16]:
print("Total columns:", df.shape[1])

new_dummy_cols = [c for c in df.columns if any(c.startswith(p + "_") for p in ["addr_state", "purpose", "home_ownership", "verification_status"])]
print("Dummy columns created:", len(new_dummy_cols))
print(new_dummy_cols[:30])

Total columns: 138
Dummy columns created: 66
['addr_state_AL', 'addr_state_AR', 'addr_state_AZ', 'addr_state_CA', 'addr_state_CO', 'addr_state_CT', 'addr_state_DC', 'addr_state_DE', 'addr_state_FL', 'addr_state_GA', 'addr_state_HI', 'addr_state_IA', 'addr_state_ID', 'addr_state_IL', 'addr_state_IN', 'addr_state_KS', 'addr_state_KY', 'addr_state_LA', 'addr_state_MA', 'addr_state_MD', 'addr_state_MI', 'addr_state_MN', 'addr_state_MO', 'addr_state_MS', 'addr_state_MT', 'addr_state_NC', 'addr_state_NE', 'addr_state_NH', 'addr_state_NJ', 'addr_state_NM']


In [17]:
df.dtypes[df.dtypes == "object"].index.tolist()

['emp_title', 'title', 'desc', 'issue_ym']

In [18]:
import os
from datetime import datetime

out_dir = r"C:\Users\ariel\Desktop\Thesis\F-TM-CR\data\03_advanced_prep"
os.makedirs(out_dir, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M")
out_path = os.path.join(out_dir, f"lc_after_03_advanced_prep_basic+test_{ts}.csv")

df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: C:\Users\ariel\Desktop\Thesis\F-TM-CR\data\03_advanced_prep\lc_after_03_advanced_prep_basic+test_20260816_1827.csv
